In [13]:
import os
import ast
import random
import pandas as pd
import numpy as np
import implicit
import joblib

from scipy.sparse import csr_matrix
from sklearn.linear_model import LinearRegression

In [14]:
df_movies = pd.read_csv("data/movies_train.csv")
print(f"Всего фильмов в базе: {len(df_movies)}")

df_movies['genres_list'] = df_movies['genres_list'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

all_genres = []
for genres in df_movies['genres_list']:
    all_genres.extend(genres)
all_genres = list(set(all_genres))
print(f"Всего жанров: {len(all_genres)}")

ratings = []
n_users = 2000
min_ratings_per_user = 20
max_ratings_per_user = 60

random.seed(42)
np.random.seed(42)

for user_id in range(1, n_users + 1):
    k = min(random.randint(2, 4), len(all_genres))
    fav_genres = random.sample(all_genres, k=k)

    candidate_movies = df_movies[
        df_movies['genres_list'].apply(lambda x: any(g in x for g in fav_genres))
    ]

    if len(candidate_movies) == 0:
        continue

    n_ratings = random.randint(min_ratings_per_user, min(max_ratings_per_user, len(candidate_movies)))

    raw = candidate_movies['votes_kp'].fillna(0).values.astype(float)
    weights = np.log1p(raw)

    if weights.sum() == 0:
        weights = None
    else:
        weights = weights / weights.sum()
        max_allowed = 1.0 / n_ratings
        guard = 0
        while weights.max() > max_allowed and guard < 20:
            weights = np.power(weights, 0.5)
            weights = weights / weights.sum()
            guard += 1
        if weights.max() > max_allowed:
            weights = None

    selected = candidate_movies.sample(
        n=n_ratings, random_state=user_id, weights=weights
    )

    for _, movie in selected.iterrows():
        base = movie['rating_kp']
        match_ratio = len(set(fav_genres) & set(movie['genres_list'])) / len(fav_genres)
        genre_bonus = (match_ratio - 0.5) * 4
        noise = np.random.normal(0, 1)
        rating = base + genre_bonus + noise
        rating = int(np.clip(round(rating), 1, 10))

        ratings.append({
            'user_id': user_id,
            'movie_id': movie['id'],
            'rating': rating,
            'title': movie['name']
        })

print()
df_ratings = pd.DataFrame(ratings)
print(f"Создано {df_ratings['user_id'].nunique()} пользователей")
print(f"Всего оценок: {len(df_ratings)}")
print(f"Уникальных оценённых фильмов: {df_ratings['movie_id'].nunique()} из {len(df_movies)}")
print(f"Средняя оценка: {df_ratings['rating'].mean():.2f}")
print(f"Средний rating_kp: {df_movies['rating_kp'].mean():.2f}")

df_ratings.to_csv("data/user_ratings_emulated.csv", index=False)
print("Сохранено: data/user_ratings_emulated.csv")

print()
display(df_ratings.head(10))

Всего фильмов в базе: 17017
Всего жанров: 23

Создано 2000 пользователей
Всего оценок: 80563
Уникальных оценённых фильмов: 15927 из 17017
Средняя оценка: 6.15
Средний rating_kp: 6.50
Сохранено: data/user_ratings_emulated.csv



,user_id,movie_id,rating,title
0,1,195394,7,Принц Персии: Пески времени
1,1,1219149,6,Шан-Чи и легенда десяти колец
2,1,41519,8,Брат
3,1,418794,7,13-й район: Ультиматум
4,1,328,7,Властелин колец: Братство кольца
5,1,261513,6,Сахара
6,1,5621,7,Багровые реки 2: Ангелы апокалипсиса
7,1,271936,7,Враг государства №1: Легенда
8,1,467974,4,Поединок
9,1,468952,6,Кулак легенды: Возвращение Чэнь Чжэня


In [15]:
REL_THRESHOLD = 8
N_HIDE_HIGH = 3
N_HIDE_EXTRA = 2

users = df_ratings['user_id'].unique()
np.random.seed(42)
candidate_test = set(np.random.choice(users, size=int(len(users) * 0.2), replace=False))

train_data, test_data, test_users = [], [], set()
for user_id, user_ratings in df_ratings.groupby('user_id'):
    if len(user_ratings) < 5:
        train_data.append(user_ratings)
        continue
    high = user_ratings[user_ratings['rating'] >= REL_THRESHOLD]
    if user_id in candidate_test and len(high) >= 2:
        hide_high = high.sample(n=min(N_HIDE_HIGH, len(high)), random_state=42)
        rest = user_ratings.drop(hide_high.index)
        hide_extra = rest.sample(n=min(N_HIDE_EXTRA, len(rest) - 1), random_state=42)
        hidden = pd.concat([hide_high, hide_extra])
        train_data.append(user_ratings.drop(hidden.index))
        test_data.append(hidden)
        test_users.add(user_id)
    else:
        train_data.append(user_ratings)

train_df = pd.concat(train_data)
test_df = pd.concat(test_data)
print(f"train: {len(train_df)}  test: {len(test_df)}  тестовых юзеров: {len(test_users)}")

user_ids = df_ratings['user_id'].unique()
movie_ids = df_ratings['movie_id'].unique()
user_to_idx = {uid: i for i, uid in enumerate(user_ids)}
movie_to_idx = {mid: i for i, mid in enumerate(movie_ids)}

row = train_df['user_id'].map(user_to_idx).values
col = train_df['movie_id'].map(movie_to_idx).values
data = train_df['rating'].values.astype(float)

matrix = csr_matrix((data, (row, col)), shape=(len(user_ids), len(movie_ids)))
model = implicit.als.AlternatingLeastSquares(factors=50, iterations=30, random_state=42)
model.fit(matrix)

u = train_df['user_id'].map(user_to_idx).values
m = train_df['movie_id'].map(movie_to_idx).values
raw = np.sum(model.user_factors[u] * model.item_factors[m], axis=1)
calib = LinearRegression().fit(raw.reshape(-1, 1), train_df['rating'].values)

os.makedirs('models/current', exist_ok=True)
joblib.dump(model, 'models/current/als_model.pkl')
joblib.dump(user_to_idx, 'models/current/user_to_idx.pkl')
joblib.dump(movie_to_idx, 'models/current/movie_to_idx.pkl')
joblib.dump(user_ids, 'models/current/user_ids.pkl')
joblib.dump(movie_ids, 'models/current/movie_ids.pkl')
joblib.dump(calib, 'models/current/als_calibrator.pkl')

train_df.to_csv('data/train_ratings.csv', index=False)
test_df.to_csv('data/test_ratings.csv', index=False)

print("Сохранено в models/current/ и data/")

train: 78705  test: 1858  тестовых юзеров: 377


100%|██████████| 30/30 [00:00<00:00, 48.02it/s]


Сохранено в models/current/ и data/
